# Retsu notebook tour

This notebook mirrors the service-free quickstart. It uses the memory backend so the examples do not require Redis or Valkey.


In [ ]:
from dataclasses import dataclass

import retsu


## Configure local capacity

Define one quantitative resource and one concurrency limit.


In [ ]:
retsu.configure(backend="memory")
retsu.define_resource("memory_mb", 512)
retsu.define_concurrency("image-transform", 2)


## Guard a function

The resource request can be dynamic. Here the memory request is read from the image object passed to the function.


In [ ]:
@dataclass
class Image:
    id: str
    size_mb: int


@retsu.guard(
    resources={"memory_mb": lambda image: image.size_mb},
    concurrency={"image-transform": 1},
)
def transform(image: Image) -> str:
    return f"transformed {image.id}"


transform(Image(id="hero", size_mb=128))


## Inspect usage

Usage returns to zero after the guarded function exits because the lease is released in a `finally` path.


In [ ]:
usage = retsu.get_usage()
usage.resources["memory_mb"].used, usage.concurrency["image-transform"].used


## Admission mode

Submit a job, let a scheduler acquire capacity, and read the result from a handle.


In [ ]:
def double(value: int) -> int:
    return value * 2


handle = retsu.submit(double, args=(21,), resources={"memory_mb": 1})
retsu.Scheduler().run_once()
handle.result(timeout=5)


Next: read the [concepts](concepts.md), [guard mode](guard-mode.md), and [admission mode](admission-mode.md) guides.
